# MusicBrainz Enrichment & Text Preparation

This notebook explores the MusicBrainz enrichment data and designs the optimal text representation for embedding.

## Goals
1. Understand what MusicBrainz data we have
2. Analyze relationship data structure
3. Design rich text representation for embedding
4. Test different enrichment strategies
5. Measure text quality and coverage

In [18]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import json

from crate_analysis import Database

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 200)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

In [19]:
# Connect to database
db = Database()
print(f"Connected to: {db.db_path}")

Connected to: /Users/pooks/Dev/crate/data/music_kb.sqlite


## 1. Explore Available Tables

In [20]:
# Get all tables
tables = db.get_tables()
print(f"Total tables: {len(tables)}\n")

# Get row counts
table_stats = []
for table_name in tables['name']:
    try:
        count = db.get_row_count(table_name)
        table_stats.append({
            'table': table_name,
            'row_count': count,
            'columns': len(db.get_table_info(table_name))
        })
    except Exception as e:
        print(f"Error with {table_name}: {e}")

stats_df = pd.DataFrame(table_stats).sort_values('row_count', ascending=False)
stats_df

Total tables: 16



,table,row_count,columns
11,master_relations,32640646,12
13,mb_master_lookup,26276365,23
10,fact_plays,2193187,25
0,artist_fact_plays,1785586,10
4,entities_fts,679814,4
6,entities_fts_content,679814,5
8,entities_fts_docsize,679814,2
2,audit_log,12563,5
9,entities_fts_idx,6637,3
7,entities_fts_data,5763,2


## 2. Fact Plays - Core Data

This is our main table. Let's see what we have:

In [21]:
# Schema
print("fact_plays schema:")
display(db.get_table_info('fact_plays'))

# Sample
print("\nSample rows:")
plays_sample = db.get_table_sample('fact_plays', 10)
display(plays_sample)

fact_plays schema:


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,1,None,0
1,1,airdate,TEXT,1,None,0
2,2,show,INTEGER,1,None,0
3,3,show_uri,TEXT,1,None,0
4,4,image_uri,TEXT,0,None,0
5,5,thumbnail_uri,TEXT,0,None,0
6,6,song,TEXT,0,None,0
7,7,track_id,TEXT,0,None,0
8,8,recording_id,TEXT,0,None,0
9,9,artist,TEXT,0,None,0



Sample rows:


,id,airdate,show,show_uri,image_uri,thumbnail_uri,song,track_id,recording_id,artist,artist_ids,album,release_id,release_group_id,labels,label_ids,release_date,rotation_status,is_local,is_request,is_live,comment,play_type,created_at,updated_at
0,3518527,2025-06-25T01:49:16-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia800307.us.archive.org/26/items/mbid-abc2013c-b852-416d-967d-d6a3f21780cf/mbid-abc2013c-b852-416d-967d-d6a3f21780cf-42265177378_thumb500.jpg,https://dn721508.ca.archive.org/0/items/mbid-abc2013c-b852-416d-967d-d6a3f21780cf/mbid-abc2013c-b852-416d-967d-d6a3f21780cf-42265177378_thumb250.jpg,How I Became a Madman,3518527,2e89c6e9-540a-4ab6-ba65-1607d8a9fc3c,Ami Taf Ra feat. Kamasi Washington,"[""c7a3e868-c6d8-4512-a5ef-f6cbe42899b0"",""0e0b62fd-f14c-4fec-b284-b3cde159880d""]",The Prophet and the Madman,abc2013c-b852-416d-967d-d6a3f21780cf,f50a44d8-ed21-4519-9293-b960b0d7501f,"[""Brainfeeder""]","[""20b3d6f9-9086-48d9-802f-5f808456a0ef""]",2025-08-22,Medium,0,0,0,"North African, LA-based singer-songwriter Ami Taf Ra has announced her debut album, The Prophet and The Madman, alongside the release of new single, ""How I Became A Madman"", featuring Kamasi Washi...",trackplay,2025-06-25T09:16:30.253Z,2025-06-25T09:16:30.253Z
1,3518526,2025-06-25T01:46:50-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia601909.us.archive.org/3/items/mbid-55e12b15-e080-42a8-af93-4301e3487a45/mbid-55e12b15-e080-42a8-af93-4301e3487a45-4504278343_thumb500.jpg,https://ia801909.us.archive.org/3/items/mbid-55e12b15-e080-42a8-af93-4301e3487a45/mbid-55e12b15-e080-42a8-af93-4301e3487a45-4504278343_thumb250.jpg,Ain’t No Mountain High Enough,3518526,b04b5028-da5b-4f05-bfe3-4f8f80396d98,Marvin Gaye & Tammi Terrell,"[""afdb7919-059d-43c1-b668-ba1d265e7e42"",""ce4582b6-aee7-4da1-b841-3c619d4fb5a5""]",Hitsville USA: The Motown Singles Collection 1959–1971,55e12b15-e080-42a8-af93-4301e3487a45,ebc5c8a7-5537-34b2-a7ad-a2bc50ba4a31,"[""Motown""]","[""8e479e57-ef44-490c-b75d-cd28df89bf1b""]",1992-11-03,None,0,0,0,"The extraordinary Tammi Terrell died just before her 25th birthday of brain cancer. Although Marvin Gaye's relationship with her was platonic, it's said that he never got over her death.: https:/...",trackplay,2025-06-25T09:16:30.254Z,2025-06-25T09:16:30.254Z
2,3518525,2025-06-25T01:42:30-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia801509.us.archive.org/34/items/mbid-950ec5b2-8dc3-41c3-beff-301c2eaa1588/mbid-950ec5b2-8dc3-41c3-beff-301c2eaa1588-22256382950_thumb500.jpg,https://ia601509.us.archive.org/34/items/mbid-950ec5b2-8dc3-41c3-beff-301c2eaa1588/mbid-950ec5b2-8dc3-41c3-beff-301c2eaa1588-22256382950_thumb250.jpg,Turn Me Around,3518525,831ddcea-19a5-4588-86d7-b093c4df9376,Mavis Staples & Bonnie Raitt,"[""0f0da09c-3940-4cb5-879f-4cea28907810"",""04f57a2d-2449-400e-8fff-5b1c5af9560b""]",I'll Take You There: An All-Star Concert Celebration,950ec5b2-8dc3-41c3-beff-301c2eaa1588,038454a4-3998-4b42-8d5e-09af2fbea082,"[""Blackbird Presents""]","[""30ec5ddd-d62a-4167-aae7-89845b21ae1d""]",2017-06-02,Library,0,0,0,From Mavis Staples: I’ll Take You There — An All-Star Concert Celebration. Chicago’s Auditorium Theater 2014 \n https://www.youtube.com/watch?v=A3r63pFUk-A,trackplay,2025-06-25T09:16:30.254Z,2025-06-25T09:16:30.254Z
3,3518524,2025-06-25T01:39:11-07:00,63830,https://api.kexp.org/v2/shows/63830/?format=json,https://ia600109.us.archive.org/22/items/mbid-8244e6f2-77f6-4a33-9482-4b6ee6cdc84d/mbid-8244e6f2-77f6-4a33-9482-4b6ee6cdc84d-41495747450_thumb500.jpg,https://ia600109.us.archive.org/22/items/mbid-8244e6f2-77f6-4a33-9482-4b6ee6cdc84d/mbid-8244e6f2-77f6-4a33-9482-4b6ee6cdc84d-41495747450_thumb250.jpg,Lovers’ Holiday,3518524,cb65609c-6a14-45cf-a178-765be6354bb3,Durand Jones & The Indications,"[""5da8d9b1-af89-43a3-a519-8e32ec21e7f5""]",Flowers,8244e6f2-77f6-4a33-9482-4b6ee6cdc84d,ef7063f4-5f5c-49ea-a1b8-24a4b5d66351,"[""Dead Oceans""]","[""f70f950f-2587-4f85-a5c7-b483a47bd2e9"

In [22]:
# Analyze field completeness
completeness = db.query("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN artist IS NOT NULL AND artist != '' THEN 1 ELSE 0 END) as has_artist,
        SUM(CASE WHEN album IS NOT NULL AND album != '' THEN 1 ELSE 0 END) as has_album,
        SUM(CASE WHEN song IS NOT NULL AND song != '' THEN 1 ELSE 0 END) as has_song,
        SUM(CASE WHEN rotation_status IS NOT NULL THEN 1 ELSE 0 END) as has_rotation,
        SUM(CASE WHEN is_local IS NOT NULL THEN 1 ELSE 0 END) as has_local,
        SUM(CASE WHEN labels IS NOT NULL AND labels != '' THEN 1 ELSE 0 END) as has_labels,
        SUM(CASE WHEN comment IS NOT NULL AND comment != '' THEN 1 ELSE 0 END) as has_comment
    FROM fact_plays
""")

total = completeness['total'].iloc[0]
completeness_pct = pd.DataFrame({
    'field': ['artist', 'album', 'song', 'rotation', 'local', 'labels', 'comment'],
    'count': [
        completeness['has_artist'].iloc[0],
        completeness['has_album'].iloc[0],
        completeness['has_song'].iloc[0],
        completeness['has_rotation'].iloc[0],
        completeness['has_local'].iloc[0],
        completeness['has_labels'].iloc[0],
        completeness['has_comment'].iloc[0]
    ]
})
completeness_pct['percentage'] = (completeness_pct['count'] / total * 100).round(2)

print(f"\nField completeness (out of {total:,} plays):")
display(completeness_pct)

# Visualize
fig = px.bar(completeness_pct, x='field', y='percentage',
             title='Field Completeness in fact_plays',
             labels={'percentage': 'Completeness %', 'field': 'Field'})
fig.add_hline(y=100, line_dash="dash", line_color="green", annotation_text="100%")
fig.show()


Field completeness (out of 2,193,187 plays):


,field,count,percentage
0,artist,2192820,99.98
1,album,2081423,94.90
2,song,2189063,99.81
3,rotation,1742875,79.47
4,local,2193187,100.00
5,labels,2186348,99.69
6,comment,1074417,48.99


## 3. Comments - Rich Metadata!

DJ comments can contain genre info, mood, descriptions - super valuable for semantic search.

In [23]:
# Get plays with comments
comments = db.query("""
    SELECT artist, song, album, comment, airdate
    FROM fact_plays
    WHERE comment IS NOT NULL AND comment != ''
    ORDER BY RANDOM()
    LIMIT 50
""")

print(f"Found {len(comments)} plays with comments (out of sample)\n")

# Show examples
for idx, row in comments.head(20).iterrows():
    print(f"🎵 {row['artist']} - {row['song']}")
    print(f"   💬 {row['comment']}")
    print()

Found 50 plays with comments (out of sample)

🎵 Purity Ring - Amenamy
   💬 Debut album "Shrines" drops next week via 4AD, and the Montreal-based duo is returning to Seattle on Wednesday, September 5th for an all ages show at Neumos with I Break Horses.

🎵 Junius Meyvant - Color Decay
   💬 Debut full length from Junius Meyvant! Listen to more on his website http://juniusmeyvant.com/ and then enter the KEXP Iceland Airwaves Flyaway contest and see him and Warpaint and PJ Harvey in beautiful Reykjavik! http://blog.kexp.org/iceland-airwaves-flyaway-2016/

🎵 Betty Davis - Hangin' Out
   💬 Ms. Davis’s unique story, still sadly mostly unknown, is unlike any other in popular music. Betty wrote the song “Uptown” for the Chambers Brothers before marrying Miles Davis in the late ‘60s, influencing him with psychedelic rock, and introducing him to Jimi Hendrix. But her songwriting ability was way ahead of its time as well. Betty not only wrote every song she ever recorded and produced every album a

In [24]:
# Analyze comment length and characteristics
comment_stats = db.query("""
    SELECT 
        LENGTH(comment) as comment_length,
        comment
    FROM fact_plays
    WHERE comment IS NOT NULL AND comment != ''
""")

print(f"Comment statistics:")
print(f"  Average length: {comment_stats['comment_length'].mean():.0f} characters")
print(f"  Median length: {comment_stats['comment_length'].median():.0f} characters")
print(f"  Max length: {comment_stats['comment_length'].max():.0f} characters")

# Distribution
fig = px.histogram(comment_stats, x='comment_length', nbins=50,
                   title='Comment Length Distribution',
                   labels={'comment_length': 'Comment Length (characters)'})
fig.show()

Comment statistics:
  Average length: 200 characters
  Median length: 137 characters
  Max length: 5575 characters


## 4. MusicBrainz Artist Entity Data

In [25]:
# Check if we have artist entity tables
artist_tables = [t for t in tables['name'].values if 'artist' in t.lower()]
print("Artist-related tables:")
for t in artist_tables:
    count = db.get_row_count(t)
    print(f"  {t}: {count:,} rows")

Artist-related tables:
  artist_fact_plays: 1,785,586 rows
  artists_masters: 0 rows
  mb_artist_unresolved: 1,213 rows


In [26]:
# Explore artist master table (if it exists)
if 'artist_mb_entity_master' in tables['name'].values:
    print("Schema:")
    display(db.get_table_info('artist_mb_entity_master'))
    
    print("\nSample:")
    artist_sample = db.get_table_sample('artist_mb_entity_master', 10)
    display(artist_sample)
    
    # Check what metadata we have
    artist_meta = db.query("""
        SELECT 
            COUNT(*) as total,
            SUM(CASE WHEN type IS NOT NULL THEN 1 ELSE 0 END) as has_type,
            SUM(CASE WHEN country IS NOT NULL THEN 1 ELSE 0 END) as has_country,
            SUM(CASE WHEN gender IS NOT NULL THEN 1 ELSE 0 END) as has_gender,
            SUM(CASE WHEN disambiguation IS NOT NULL THEN 1 ELSE 0 END) as has_disambiguation,
            COUNT(DISTINCT type) as unique_types,
            COUNT(DISTINCT country) as unique_countries
        FROM artist_mb_entity_master
    """)
    print("\nMetadata completeness:")
    display(artist_meta)
    
    # Get artist types distribution
    artist_types = db.query("""
        SELECT type, COUNT(*) as count
        FROM artist_mb_entity_master
        WHERE type IS NOT NULL
        GROUP BY type
        ORDER BY count DESC
    """)
    print("\nArtist types:")
    display(artist_types)

## 5. Relationships - Genre, Labels, etc.

In [27]:
# Check for relationship tables
rel_tables = [t for t in tables['name'].values if 'relation' in t.lower() or 'master' in t.lower()]
print("Relationship tables:")
for t in rel_tables:
    count = db.get_row_count(t)
    print(f"  {t}: {count:,} rows")

Relationship tables:
  artists_masters: 0 rows
  master_relations: 32,640,646 rows
  mb_master_lookup: 26,276,365 rows


In [28]:
# Explore master_relations if it exists
if 'master_relations' in tables['name'].values:
    print("Schema:")
    display(db.get_table_info('master_relations'))
    
    print("\nSample:")
    rel_sample = db.get_table_sample('master_relations', 20)
    display(rel_sample)
    
    # Get relationship type distribution
    rel_types = db.query("""
        SELECT 
            predicate,
            COUNT(*) as count,
            COUNT(DISTINCT subject_id) as unique_subjects,
            COUNT(DISTINCT object_id) as unique_objects
        FROM master_relations
        GROUP BY predicate
        ORDER BY count DESC
    """)
    print("\nRelationship types:")
    display(rel_types)
    
    # Visualize
    fig = px.bar(rel_types, x='predicate', y='count',
                 title='Relationship Type Distribution',
                 labels={'count': 'Count', 'predicate': 'Relationship Type'})
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,subject_id,TEXT,1,None,0
1,1,subject_type,TEXT,1,None,0
2,2,subject_name,TEXT,0,None,0
3,3,predicate,TEXT,1,None,0
4,4,object_id,TEXT,1,None,0
5,5,object_type,TEXT,1,None,0
6,6,object_name,TEXT,0,None,0
7,7,attribute_type,TEXT,0,None,0
8,8,source,TEXT,1,None,0
9,9,kexp_play_id,INTEGER,0,None,0



Sample:


,subject_id,subject_type,subject_name,predicate,object_id,object_type,object_name,attribute_type,source,kexp_play_id,updated_at,created_at
0,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,d27fdd10-e1df-4208-8e2e-62187866246f,recording,Alright (radio version),,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
1,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,16a4347d-1ed8-4b73-b372-9a373d156c41,recording,Fly,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
2,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,47985ebe-998e-4198-878d-c241258d2c3a,recording,Fly,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
3,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,789d456b-f206-4804-bd6e-ddbe40fb6e3f,recording,Fly,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
4,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,f9f77ccc-fd3b-4b1c-af98-be5f708b25cd,recording,Na Na Na Na,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
5,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,performer,7ac765b7-cb48-4cd0-af23-f0e5b1f0a8b5,recording,Scalp Dem,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
6,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,producer,995d7811-c420-4066-b9bb-b53cb44cfc3c,recording,Dolly My Baby,,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
7,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,vocal,d89b9792-7135-44ca-a3cd-da8556326bd9,recording,Fly,additional,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
8,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,vocal,16a4347d-1ed8-4b73-b372-9a373d156c41,recording,Fly,additional,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12
9,2d5fbbfd-27a7-4b74-848a-2b1f24fa1d0a,artist,Super Cat,vocal,56a3b1b1-1210-49a2-abf1-0d53571de1ed,recording,The Don of Dons (Put de Ting Pon Dem),,musicbrainz,None,2025-06-26 22:11:12,2025-06-26 22:11:12



Relationship types:


,predicate,count,unique_subjects,unique_objects
0,instrument,6576850,297217,1901239
1,vocal,2633697,210110,1571258
2,producer,2244680,132207,1620025
3,composer,2088189,174139,1734218
4,has_artist,1743633,1673390,65380
...,...,...,...,...
108,adapter,97,36,92
109,named after release group,87,81,84
110,video copyright,59,11,58
111,named after label,16,16,16


In [ ]:
# Get entity type distribution
if 'master_relations' in tables['name'].values:
    entity_types = db.query("""
        SELECT 
            subject_type,
            object_type,
            COUNT(*) as count
        FROM master_relations
        GROUP BY subject_type, object_type
        ORDER BY count DESC
    """)
    print("Entity type combinations:")
    display(entity_types.head(20))

## 6. Example: Rich Artist Profile

Let's pick a random artist and see all the data we can gather about them:

In [29]:
# Pick a popular artist from plays
popular_artist = db.query("""
    SELECT artist, COUNT(*) as play_count
    FROM fact_plays
    WHERE artist IS NOT NULL
    GROUP BY artist
    ORDER BY play_count DESC
    LIMIT 1
""").iloc[0]['artist']

print(f"Building profile for: {popular_artist}\n")

# Get artist entity data
if 'artist_mb_entity_master' in tables['name'].values:
    artist_entity = db.query("""
        SELECT *
        FROM artist_mb_entity_master
        WHERE name = ?
        LIMIT 1
    """, (popular_artist,))
    
    if not artist_entity.empty:
        print("MusicBrainz Entity Data:")
        display(artist_entity.T)

# Get relationships
if 'master_relations' in tables['name'].values:
    artist_rels = db.query("""
        SELECT 
            predicate,
            object_name,
            object_type
        FROM master_relations
        WHERE subject_name = ?
        LIMIT 50
    """, (popular_artist,))
    
    if not artist_rels.empty:
        print(f"\nRelationships ({len(artist_rels)} found):")
        display(artist_rels)

# Get sample plays with comments
artist_plays = db.query("""
    SELECT song, album, comment, labels, rotation_status
    FROM fact_plays
    WHERE artist = ?
    AND comment IS NOT NULL AND comment != ''
    LIMIT 10
""", (popular_artist,))

if not artist_plays.empty:
    print(f"\nSample plays with comments:")
    display(artist_plays)

Building profile for: Radiohead


Relationships (50 found):


,predicate,object_name,object_type
0,instrument arranger,Airbag,recording
1,instrument arranger,Airbag,recording
2,instrument arranger,Climbing Up the Walls,recording
3,instrument arranger,Electioneering,recording
4,instrument arranger,Exit Music (for a Film),recording
5,instrument arranger,Fitter Happier,recording
6,instrument arranger,Karma Police,recording
7,instrument arranger,Let Down,recording
8,instrument arranger,Lucky,recording
9,instrument arranger,No Surprises,recording



Sample plays with comments:


,song,album,comment,labels,rotation_status
0,Go to Sleep. (Little Man Being Erased.),Hail to the Thief. (The Gloaming.),"Goodnight John and Morgan! \n\nFor everyone else, welcome to the One-Day Summer Drive on the Midday Show! Show your support for KEXP at https://www.kexp.org/","[""Capitol Records""]",None
1,Weird Fishes/Arpeggi,In Rainbows,"Happy Birthday to Brandie (5/31)!\n\n""Weird Fishes/Arpeggi"" comes from Radiohead's seventh full-length album.\n\nThe Smile, featuring Radiohead's Thom Yorke and Jonny Greenwood, stopped by KEXP in...","[""TBD Records""]",None
2,Reckoner,In Rainbows,"This song, formerly known as ""Feeling Pulled Apart By Horses,"" was debuted by the band at the Gorge Amphitheatre in George, Washington, on June 23, 2001.\n--\nThom Yorke has said the guitar riff w...","[""TBD Records""]",None
3,A Reminder,OK Computer: OKNOTOK 1997 2017,"Radiohead recorded this track along with the rest of OK Computer, but was left off the album and released as a b-side to Paranoid Android instead.","[""XL Recordings""]",Library
4,There There. (The Boney King of Nowhere.),Hail to the Thief. (The Gloaming.),"Thom Yorke on this song: \n""It made me cry when we finished it, actually, I blubbed my eyes out. I just thought it was the best thing we'd ever done. What I discovered, I think, in making this re...","[""Capitol Records""]",None
5,Stop Whispering,Pablo Honey,"When asked to headline a show with The Pixies opening, Radiohead refused to headline, saying ""That's like the Beatles opening for us."" This song is a tribute to the band Pixies, who were a big inf...","[""Capitol Records""]",None
6,Black Star,The Bends,Embry said this is the album of the year,"[""EMI Music Canada"",""Parlophone""]",None
7,15 Step,In Rainbows,"The Devil is Major Arcana card XV (15), and in numerology, 15 reduces to 6 (1 + 5 = 6), which is the number of The Lovers card.","[""TBD Records""]",None
8,Climbing Up the Walls,OK Computer,"Celebrating the release of OK Computer 28 years ago today!\n\nRecorded by the band at St. Catherine's Court, a manor house located in Bath, England that was built in the 16th century. That probabl...","[""Capitol Records""]",None
9,Electioneering,OK Computer,"Playing the whole album OK Computer in celebration of its release on this day in 1997!\n\nThom Yorke explained in an interview in 1997 that ""Electioneering"" was inspired by the writings of the med...","[""Capitol Records""]",None


## 7. Design Text Enrichment Strategy

Based on what we found, let's design the optimal text representation:

In [30]:
def enrich_play_text(play_row, relationships_df=None):
    """Create rich text representation for a play."""
    parts = []
    
    # Core: Artist - Song - Album
    if play_row.get('artist'):
        parts.append(f"{play_row['artist']}")
    if play_row.get('song'):
        parts.append(f"- {play_row['song']}")
    if play_row.get('album'):
        parts.append(f"- {play_row['album']}")
    
    metadata_parts = []
    
    # Add DJ comment (valuable!)
    if play_row.get('comment'):
        metadata_parts.append(f"Comment: {play_row['comment']}")
    
    # Add labels
    if play_row.get('labels'):
        metadata_parts.append(f"Label: {play_row['labels']}")
    
    # Add rotation status
    if play_row.get('rotation_status'):
        metadata_parts.append(f"Rotation: {play_row['rotation_status']}")
    
    # Add local flag
    if play_row.get('is_local') == 1:
        metadata_parts.append("Local artist")
    
    # Add year if available
    if play_row.get('airdate'):
        year = pd.to_datetime(play_row['airdate']).year
        metadata_parts.append(f"Year: {year}")
    
    # Add MusicBrainz relationships (genres, etc.) if provided
    if relationships_df is not None and not relationships_df.empty:
        # Extract genres from relationships
        genres = relationships_df[
            (relationships_df['predicate'].str.contains('genre', case=False, na=False))
        ]['object_name'].tolist()
        if genres:
            metadata_parts.append(f"Genre: {', '.join(genres[:5])}")  # Limit to 5
    
    # Combine
    text = ' '.join(parts)
    if metadata_parts:
        text += ' | ' + ' | '.join(metadata_parts)
    
    return text

# Test on sample plays
test_sample = db.query("""
    SELECT *
    FROM fact_plays
    LIMIT 10
""")

print("Example enriched texts:\n")
for idx, row in test_sample.iterrows():
    enriched = enrich_play_text(row)
    print(f"{idx+1}. {enriched}")
    print()

Example enriched texts:

1. Ami Taf Ra feat. Kamasi Washington - How I Became a Madman - The Prophet and the Madman | Comment: North African, LA-based singer-songwriter Ami Taf Ra has announced her debut album, The Prophet and The Madman, alongside the release of new single, "How I Became A Madman", featuring Kamasi Washington. | Label: ["Brainfeeder"] | Rotation: Medium | Year: 2025

2. Marvin Gaye & Tammi Terrell - Ain’t No Mountain High Enough - Hitsville USA: The Motown Singles Collection 1959–1971 | Comment: The extraordinary Tammi Terrell died just before her 25th birthday of brain cancer.  Although Marvin Gaye's relationship with her was platonic, it's said that he never got over her death.: https://www.smoothradio.com/news/music/tammi-terrell-songs-death-children-age/ | Label: ["Motown"] | Year: 2025

3. Mavis Staples & Bonnie Raitt - Turn Me Around - I'll Take You There: An All-Star Concert Celebration | Comment: From Mavis Staples: I’ll Take You There — An All-Star Concert Ce

## 8. Analyze Text Coverage & Quality

In [31]:
# Get a larger sample and analyze enrichment quality
sample_size = 1000
sample_plays = db.query(f"""
    SELECT *
    FROM fact_plays
    ORDER BY RANDOM()
    LIMIT {sample_size}
""")

# Generate enriched texts
enriched_texts = []
for idx, row in sample_plays.iterrows():
    text = enrich_play_text(row)
    enriched_texts.append({
        'text': text,
        'length': len(text),
        'has_comment': bool(row.get('comment')),
        'has_label': bool(row.get('labels')),
        'has_rotation': bool(row.get('rotation_status'))
    })

enriched_df = pd.DataFrame(enriched_texts)

print(f"Analysis of {sample_size} enriched texts:\n")
print(f"Average length: {enriched_df['length'].mean():.0f} characters")
print(f"Median length: {enriched_df['length'].median():.0f} characters")
print(f"% with comments: {enriched_df['has_comment'].mean()*100:.1f}%")
print(f"% with labels: {enriched_df['has_label'].mean()*100:.1f}%")
print(f"% with rotation: {enriched_df['has_rotation'].mean()*100:.1f}%")

# Show distribution
fig = px.histogram(enriched_df, x='length', nbins=50,
                   title=f'Enriched Text Length Distribution (n={sample_size})',
                   labels={'length': 'Text Length (characters)'})
fig.show()

Analysis of 1000 enriched texts:

Average length: 206 characters
Median length: 134 characters
% with comments: 47.4%
% with labels: 99.7%
% with rotation: 78.6%


In [ ]:
# Show examples by richness
print("\n=== Most Enriched Examples ===")
for idx in enriched_df.nlargest(5, 'length').index:
    print(f"\n{enriched_df.loc[idx, 'text']}")
    print(f"  Length: {enriched_df.loc[idx, 'length']} chars")

print("\n=== Least Enriched Examples ===")
for idx in enriched_df.nsmallest(5, 'length').index:
    print(f"\n{enriched_df.loc[idx, 'text']}")
    print(f"  Length: {enriched_df.loc[idx, 'length']} chars")

## 9. Extract Genre Information

Let's see if we can extract genre info from relationships or comments:

In [32]:
# Music genre keywords to search for in comments
genre_keywords = [
    'rock', 'indie', 'electronic', 'jazz', 'classical', 'hip-hop', 'rap',
    'metal', 'punk', 'folk', 'country', 'blues', 'soul', 'funk', 'disco',
    'house', 'techno', 'ambient', 'experimental', 'pop', 'r&b', 'reggae',
    'alternative', 'grunge', 'shoegaze', 'post-punk', 'synth', 'psychedelic'
]

# Find genre mentions in comments
comment_genres = db.query("""
    SELECT comment, artist, song
    FROM fact_plays
    WHERE comment IS NOT NULL
    LIMIT 1000
""")

genre_matches = []
for idx, row in comment_genres.iterrows():
    comment_lower = row['comment'].lower()
    found_genres = [g for g in genre_keywords if g in comment_lower]
    if found_genres:
        genre_matches.append({
            'artist': row['artist'],
            'song': row['song'],
            'comment': row['comment'],
            'genres_found': found_genres
        })

print(f"Found {len(genre_matches)} comments with genre mentions (out of {len(comment_genres)} with comments)\n")

# Show examples
for match in genre_matches[:10]:
    print(f"🎵 {match['artist']} - {match['song']}")
    print(f"   Genres: {', '.join(match['genres_found'])}")
    print(f"   💬 {match['comment']}")
    print()

Found 342 comments with genre mentions (out of 1000 with comments)

🎵 SPRINTS - Descartes
   Genres: punk
   💬 The Irish punks will follow-up last year’s excellent "Letter To Self" with a record that hears the band ‘trying to make sense of a society gone mad." "All That Is Over" will be out on September 26th.
--
“Descartes was written on a plane while reading "Outline" by Rachel Cusk,” explains Karla Chubb. “Sparked directly from the line, ‘Vanity is the curse of our culture,’ the rest of the song spilled out of me quite instantaneously. Descartes explores the idea of needing to write and create as a means of survival, not just a means of expression, and how important that is to me as a person to try to understand and process the world around me.”

🎵 Penny Penny - Shilungu
   Genres: house
   💬 Penny Penny's 2001 impossibly rare Kwaito House monster "Shilungu”.

A hypnotic, percussive, groove-driven anthem, it features chanting in Tsonga, celebrating South African icon Penny Penny’s Sh

## 10. Save Enrichment Function

Let's save our enrichment function to use in the main processing:

In [ ]:
# Save to src/crate_analysis/enrichment.py
enrichment_code = '''
"""Text enrichment utilities for music search."""

import pandas as pd
from typing import Optional, Dict, Any


def enrich_play_text(play_row: Dict[str, Any], relationships_df: Optional[pd.DataFrame] = None) -> str:
    """Create rich text representation for a play.
    
    Args:
        play_row: Dictionary with play data (artist, song, album, comment, etc.)
        relationships_df: Optional DataFrame with MusicBrainz relationships
        
    Returns:
        Enriched text string ready for embedding
    """
    parts = []
    
    # Core: Artist - Song - Album
    if play_row.get('artist'):
        parts.append(f"{play_row['artist']}")
    if play_row.get('song'):
        parts.append(f"- {play_row['song']}")
    if play_row.get('album'):
        parts.append(f"- {play_row['album']}")
    
    metadata_parts = []
    
    # Add DJ comment (very valuable for semantic understanding!)
    if play_row.get('comment'):
        metadata_parts.append(f"Comment: {play_row['comment']}")
    
    # Add labels
    if play_row.get('labels'):
        metadata_parts.append(f"Label: {play_row['labels']}")
    
    # Add rotation status
    if play_row.get('rotation_status'):
        metadata_parts.append(f"Rotation: {play_row['rotation_status']}")
    
    # Add local flag
    if play_row.get('is_local') == 1:
        metadata_parts.append("Local artist")
    
    # Add year if available
    if play_row.get('airdate'):
        try:
            year = pd.to_datetime(play_row['airdate']).year
            metadata_parts.append(f"Year: {year}")
        except:
            pass
    
    # Add MusicBrainz relationships (genres, etc.) if provided
    if relationships_df is not None and not relationships_df.empty:
        # Extract genres from relationships
        genres = relationships_df[
            (relationships_df['predicate'].str.contains('genre', case=False, na=False))
        ]['object_name'].tolist()
        if genres:
            metadata_parts.append(f"Genre: {', '.join(genres[:5])}")  # Limit to 5
    
    # Combine
    text = ' '.join(parts)
    if metadata_parts:
        text += ' | ' + ' | '.join(metadata_parts)
    
    return text
'''

with open('../src/crate_analysis/enrichment.py', 'w') as f:
    f.write(enrichment_code)

print("✅ Saved enrichment function to src/crate_analysis/enrichment.py")

## Summary & Next Steps

### What We Found
1. **Comments are gold** - DJ comments contain genre, mood, style descriptions
2. **Labels are useful** - Record labels help with style/genre inference
3. **Rotation status** - Heavy/Medium/Light rotation indicates popularity/quality
4. **MusicBrainz relationships** - If available, provide structured genre data
5. **Coverage varies** - Not all plays have rich metadata

### Enrichment Strategy
- Base: Artist - Song - Album
- Add: DJ comments (when available)
- Add: Labels, rotation status, local flag
- Add: Year from airdate
- Add: MusicBrainz genres (if in relationships)

### Next Steps
1. Test embedding with enriched text
2. Compare different enrichment levels
3. Process full dataset
4. Build search interface

In [ ]:
# Clean up
db.close()